# Make Kinesis exactly-once with a DynamoDB conditional write, then prove it under a chaos injection

Your clickstream producer team finally admitted what you already suspected: the network is flaky, the SDK retries on timeout, and the same record can land on the stream two or three times. The sink has been double-counting orders for weeks. Reporting noticed last quarter. Finance noticed last Friday.

Today you build the consumer that does not care how many times the producer retries. The same `event_id` lands in the sink exactly once. Then you run the chaos injection that proves it: 1,000 events, 300 deliberate duplicates, exactly 700 rows in the Iceberg sink, 300 rejected conditional writes in the CloudWatch log. By the end of the lab you can write the post-mortem the next time a producer team forgets to mention they enabled retries.

The pattern is the same one every streaming pipeline at scale eventually adopts:

1. Producer emits a deterministic `event_id` per business event.
2. Consumer attempts a DynamoDB `put_item` with `ConditionExpression="attribute_not_exists(event_id)"` before writing to the sink.
3. On `ConditionalCheckFailedException`, the consumer logs the rejection and skips the sink write.
4. Sink write happens only after the DynamoDB write succeeds. Order matters; the alternative is orphan rows in the sink.

Four checkpoints map to four deliverables:

1. Producer emits 1,000 records to Kinesis: 700 unique `event_id` values, 300 intentional duplicates. The producer helper is provided. You do not edit it because the chaos ratio is fixed for the lab.
2. DynamoDB dedupe table contains exactly 700 unique `event_id` rows after the consumer drains the stream.
3. Iceberg sink (`events_iceberg`) contains exactly 700 rows, no duplicate `event_id` values, queried via Athena.
4. Consumer CloudWatch log shows the `ConditionalCheckFailedException` rejection line ~300 times over the run window. Log query via CloudWatch Logs Insights.

**Time.** About 75 minutes hands-on inside a 90-minute session window. The Lambda IAM role propagation wait, the producer emission, and the consumer drain dominate wall-clock; the rest is short.

**Cost.** ONE HOURLY METER. The Kinesis Data Stream bills $0.015 per shard-hour for 1 shard from the moment `create_stream` returns until cleanup deletes it. Idle shards still bill. Everything else is pay-per-request or free at lab volume. DynamoDB on-demand: 1,000 writes is roughly one tenth of a cent. Lambda: well inside the always-free 1M-requests + 400k GB-s monthly tier. The expensive line item is the Iceberg INSERTs the consumer fires through Athena: 1,000 small INSERT queries at fractions of a penny each add up. Typical session cost is $1.00 to $2.00. The cleanup cell tears down the stream first (highest hourly rate; critical-tagged) so the meter stops the moment delete returns. Set the $20 per month billing alert in your AWS console before you start.

This lab is sub-track C2 in the Data Engineering track (Streaming & CDC). Differentiation versus DEA-C01: Lab 3 is a Firehose ingestion lab, Lab 8 wires Kinesis to Redshift; neither exercises exactly-once under chaos. Interview talking point: "I built a Kinesis pipeline that survives a 30 percent duplicate rate from the producer with zero sink-side duplicates."


In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned to a specific version per LAB-CREATION-BLUEPRINT
# build rules. Never use unpinned installs.

!pip install --quiet labrun-checks==0.8.0


In [ ]:
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT
# section 12. Lab 11 has one hourly-billed resource: the 1-shard Kinesis
# Data Stream at $0.015 per shard-hour. DynamoDB on-demand is pay-per-request
# and not critical. The Lambda consumer is free at lab volume.

import atexit
import getpass
import hashlib
import io
import json
import random
import sys
import time
import uuid
import zipfile
from datetime import datetime, timedelta, timezone

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
    CheckpointResult,
)

LAB_ID = "kinesis-exactly-once"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID  # must equal cert YAML labs[].slug exactly
REGION = "us-east-1"  # all track-data-engineering labs run in us-east-1 per LAB-CREATION-BLUEPRINT section 15

# Resource names.
STREAM_NAME = f"labrun-{LAB_ID}-stream"
DEDUPE_TABLE = f"labrun-{LAB_ID}-dedupe"
CONSUMER_ROLE_NAME = f"labrun-{LAB_ID}-consumer-role"
CONSUMER_LAMBDA_NAME = f"labrun-{LAB_ID}-consumer"
GLUE_DB = f"labrun-{LAB_ID}-db"
ICEBERG_TABLE = "events_iceberg"
BUCKET_NAME = None  # resolved after STS once the account ID is known
EVENT_MAPPING_UUID = None  # set in Task 2 once create_event_source_mapping returns

# Stream shape. Exactly 1 shard per RESOURCE-SAFETY-SPEC Lab 8 pattern; we
# inherit the same one-shard rule so the meter stays at $0.015/shard-hour.
SHARD_COUNT = 1

# Chaos injection shape. Deterministic so row counts are stable across runs.
# 700 unique event_ids, 300 deliberate duplicates resampled from the unique
# pool, 1000 total records on the stream.
SEED = 42
UNIQUE_EVENT_COUNT = 700
DUPLICATE_EVENT_COUNT = 300
TOTAL_EVENT_COUNT = UNIQUE_EVENT_COUNT + DUPLICATE_EVENT_COUNT  # 1000

# Producer log object (used by Checkpoint 1).
PRODUCER_LOG_KEY = "producer/log.json"

# Iceberg sink batch id (carried inside each record).
RUN_BATCH_ID = f"batch-{int(time.time())}"

# Lambda runtime + handler.
LAMBDA_RUNTIME = "python3.13"
LAMBDA_HANDLER = "index.handler"


In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# and confirm the region. This cell must succeed before the manifest cell
# creates anything per LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")
print(f"Session token in use: {bool(aws_session_token)}")

BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
print(f"Bucket name resolved: {BUCKET_NAME}")

# Register the session with labrun-checks. register() returns None;
# do not assign its return value.
register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")


In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
#
# Per RESOURCE-SAFETY-SPEC Rule 2, the Kinesis stream is the one hourly-billed
# resource and tears down FIRST so the meter stops even if the rest of cleanup
# fails. The event-source mapping unwires the consumer before the function
# itself is deleted (otherwise Lambda surfaces an in-use error on the mapping).
# DynamoDB is on-demand so it has no idle cost, but it still goes before the
# Iceberg table because the consumer ordering writes to DynamoDB first; the
# manifest mirrors that ordering in reverse. labrun-checks 0.8.0 ships an
# iceberg_table adapter (DROP TABLE via Athena); the inline _LabAdapter
# below predates that adapter and is now functionally equivalent to the
# default AwsCleanupAdapter, so it can be dropped in a follow-up.
#
# Per RESOURCE-SAFETY-SPEC Rule 4, the orphan scan blocks execution if any
# tagged resources from a prior session exist (not just print a warning).

CLEANUP_MANIFEST = [
    CleanupResource(
        type="kinesis_stream",
        id=STREAM_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws kinesis delete-stream --stream-name {STREAM_NAME} "
            f"--enforce-consumer-deletion"
        ),
        critical=True,
    ),
    CleanupResource(
        type="lambda_event_source_mapping",
        id="(set after create_event_source_mapping)",
        region=REGION,
        cli_delete_command=(
            "aws lambda delete-event-source-mapping --uuid <uuid set in Task 2>"
        ),
    ),
    CleanupResource(
        type="lambda_function",
        id=CONSUMER_LAMBDA_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws lambda delete-function --function-name {CONSUMER_LAMBDA_NAME}"
        ),
    ),
    CleanupResource(
        type="dynamodb_table",
        id=DEDUPE_TABLE,
        region=REGION,
        cli_delete_command=f"aws dynamodb delete-table --table-name {DEDUPE_TABLE}",
    ),
    CleanupResource(
        type="iceberg_table",
        id=f"{GLUE_DB}.{ICEBERG_TABLE}",
        region=REGION,
        cli_delete_command=(
            f"aws athena start-query-execution "
            f"--query-string 'DROP TABLE IF EXISTS {GLUE_DB}.{ICEBERG_TABLE}'"
        ),
    ),
    CleanupResource(
        type="iam_role",
        id=CONSUMER_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {CONSUMER_ROLE_NAME}",
    ),
    CleanupResource(
        type="glue_database",
        id=GLUE_DB,
        region=REGION,
        cli_delete_command=f"aws glue delete-database --name {GLUE_DB}",
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
]


def _atexit_cleanup() -> None:
    """Best-effort teardown on kernel shutdown.

    The dedicated cleanup cell is the authoritative entry point; this is
    the safety net for kernel crashes during the consumer-drain wait or
    any other long-running step. Errors are swallowed because atexit
    handlers must not raise.
    """
    try:
        if "_LabAdapter" in globals():
            run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter())
        else:
            run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    """Refuse to start if a previous run left tagged resources behind.

    Per RESOURCE-SAFETY-SPEC Rule 4, the setup cell must check for leftover
    state with the lab's tag and surface the cleanup command before
    creating any new resources. Critical for this lab because a leftover
    Kinesis shard is $0.015 per hour (~$2.52 per week).
    """
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns: list[str] = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Re-running")
    print("setup against an unclean state can produce a second Kinesis shard")
    print("billing in parallel with the first. Run the cleanup cell at the")
    print("bottom of this notebook first, or remove them manually with the")
    print("AWS CLI commands the cleanup cell prints, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")


## Task 1: Stand up the stream, the bucket, and the Iceberg sink table, then run the producer that emits 1,000 records with a 30 percent duplicate ratio

The chaos starts here. The producer helper below (`run_producer`) emits exactly 1,000 records to the Kinesis stream:

- 700 records with unique deterministic `event_id` values (UUID5 derived from a stable namespace plus a counter).
- 300 records sampled from the same 700 IDs (the deliberate duplicates the producer "accidentally" re-emits).
- Records are JSON: `{event_id, customer_id, amount_cents, event_timestamp, batch_id}`.
- The producer writes a log object to `s3://BUCKET/producer/log.json` after the emission finishes: `{"emitted": 1000, "unique": 700, "duplicates": 300, "ids": [...700 unique ids...]}`.

You do not edit the producer. The chaos ratio is fixed for the lab; that is the entire pedagogical point. Your job in this cell is everything around the producer:

1. **Create the S3 bucket.** Region is `us-east-1`, so do not pass `LocationConstraint`. Tag it.
2. **Create the Kinesis Data Stream.** Exactly 1 shard. Tag it. Wait for `ACTIVE`.
3. **Create the Glue database** (`labrun-kinesis-exactly-once-db`).
4. **Create the Iceberg sink table** via Athena: `events_iceberg(event_id string, customer_id int, amount_cents int, event_timestamp string, batch_id string)`, format `ICEBERG`, location `s3://BUCKET/events_iceberg/`. The CREATE TABLE statement is provided as a string; you fire it via `start_query_execution` and poll until `SUCCEEDED`.
5. **Call `run_producer()`** which is defined further down the cell. It does the rest.

Checkpoint 1 reads the producer log object out of S3 and asserts `emitted == 1000`, `len(unique ids) == 700`, and `duplicates == 300`.


In [ ]:
# NBVAL_SKIP
# Task 1: stream, bucket, Iceberg sink, then run the chaos producer.
# The producer is provided. You write everything around it.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
kinesis = boto3.client(
    "kinesis",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
glue = boto3.client(
    "glue",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
athena = boto3.client(
    "athena",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

ATHENA_OUTPUT = f"s3://{BUCKET_NAME}/athena-results/"


def run_athena(query: str, database: str | None = None) -> str:
    """Fire an Athena query, wait for terminal state, return the query id.

    Raises RuntimeError on FAILED or CANCELLED so the caller sees the failure
    immediately rather than continuing against a partial state.
    """
    ctx = {"Database": database} if database else None
    kwargs = {
        "QueryString": query,
        "ResultConfiguration": {"OutputLocation": ATHENA_OUTPUT},
    }
    if ctx:
        kwargs["QueryExecutionContext"] = ctx
    resp = athena.start_query_execution(**kwargs)
    qid = resp["QueryExecutionId"]
    deadline = time.time() + 120
    while time.time() < deadline:
        desc = athena.get_query_execution(QueryExecutionId=qid)
        state = desc["QueryExecution"]["Status"]["State"]
        if state == "SUCCEEDED":
            return qid
        if state in ("FAILED", "CANCELLED"):
            reason = desc["QueryExecution"]["Status"].get("StateChangeReason", "")
            raise RuntimeError(f"Athena query {state}: {reason}\nQuery: {query}")
        time.sleep(1.5)
    raise RuntimeError(f"Athena query timed out after 120s. Query: {query}")


# YOUR CODE: create the S3 bucket (us-east-1, no LocationConstraint) and tag
# it with {LAB_TAG_KEY: LAB_TAG_VALUE} via put_bucket_tagging. Hint 3 has the
# exact two calls.

# YOUR CODE: create the Kinesis Data Stream with ShardCount=SHARD_COUNT and
# Tags={LAB_TAG_KEY: LAB_TAG_VALUE}, then wait for ACTIVE via
# kinesis.get_waiter("stream_exists") (covers ~30 seconds).

# YOUR CODE: create the Glue database GLUE_DB. The DatabaseInput dict needs
# a Name and a LocationUri pointing at s3://BUCKET_NAME/.

# YOUR CODE: create the Iceberg sink via Athena. The CREATE TABLE statement
# is provided below as CREATE_ICEBERG_SQL. Fire it via run_athena() with the
# database=GLUE_DB argument so the table lands in the right namespace.

CREATE_ICEBERG_SQL = f"""
CREATE TABLE IF NOT EXISTS {ICEBERG_TABLE} (
    event_id string,
    customer_id int,
    amount_cents int,
    event_timestamp string,
    batch_id string
)
LOCATION 's3://{BUCKET_NAME}/{ICEBERG_TABLE}/'
TBLPROPERTIES (
    'table_type' = 'ICEBERG',
    'format' = 'parquet'
)
""".strip()


# ---------------------------------------------------------------------------
# Provided producer (do NOT edit; chaos ratio is fixed for the lab).
# ---------------------------------------------------------------------------

def _deterministic_event_ids(seed: int, count: int) -> list[str]:
    rng = random.Random(seed)
    ns = uuid.UUID("12345678-1234-5678-1234-567812345678")
    ids: list[str] = []
    for i in range(count):
        seed_bytes = f"labrun-{LAB_ID}-{i}-{rng.random()}".encode()
        digest = hashlib.sha1(seed_bytes).hexdigest()[:32]
        # Format as UUID-shaped string for readability in logs.
        ids.append(
            f"{digest[0:8]}-{digest[8:12]}-{digest[12:16]}-{digest[16:20]}-{digest[20:32]}"
        )
    return ids


def run_producer() -> dict:
    """Emit 1,000 records to STREAM_NAME: 700 unique + 300 duplicates.

    Writes a log object to s3://BUCKET_NAME/producer/log.json. The log is
    what Checkpoint 1 reads.
    """
    print(f"Producer starting against stream {STREAM_NAME}...")
    unique_ids = _deterministic_event_ids(SEED, UNIQUE_EVENT_COUNT)
    rng = random.Random(SEED)

    duplicate_ids = [rng.choice(unique_ids) for _ in range(DUPLICATE_EVENT_COUNT)]
    all_ids = unique_ids + duplicate_ids
    rng.shuffle(all_ids)

    base_ts = int(time.time())
    records_payload: list[dict] = []
    for ix, event_id in enumerate(all_ids):
        records_payload.append({
            "event_id": event_id,
            "customer_id": rng.randint(1, 5000),
            "amount_cents": rng.randint(99, 99999),
            "event_timestamp": str(base_ts + ix),
            "batch_id": RUN_BATCH_ID,
        })

    # put_records caps at 500 per call. Chunk in 500s.
    chunk_size = 500
    failed = 0
    sent = 0
    for start in range(0, len(records_payload), chunk_size):
        chunk = records_payload[start:start + chunk_size]
        kinesis_records = [
            {
                "Data": json.dumps(rec).encode(),
                "PartitionKey": rec["event_id"],
            }
            for rec in chunk
        ]
        resp = kinesis.put_records(StreamName=STREAM_NAME, Records=kinesis_records)
        failed += int(resp.get("FailedRecordCount", 0))
        sent += len(chunk) - int(resp.get("FailedRecordCount", 0))
        print(f"  chunk {start}..{start + len(chunk)}: failed={resp.get('FailedRecordCount', 0)}")

    log_body = {
        "emitted": sent,
        "unique": len(set(unique_ids)),
        "duplicates": DUPLICATE_EVENT_COUNT,
        "ids": unique_ids,
        "failed": failed,
        "batch_id": RUN_BATCH_ID,
    }
    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=PRODUCER_LOG_KEY,
        Body=json.dumps(log_body).encode(),
        ContentType="application/json",
    )
    print(f"Producer finished. emitted={sent}, unique={UNIQUE_EVENT_COUNT}, duplicates={DUPLICATE_EVENT_COUNT}, failed={failed}")
    print(f"Producer log written: s3://{BUCKET_NAME}/{PRODUCER_LOG_KEY}")
    return log_body


# YOUR CODE: call run_producer() now that the stream + bucket + Iceberg sink
# exist. It returns the log dict; you can ignore the return value.


In [ ]:
# NBVAL_SKIP
# Checkpoint 1: producer log object exists at s3://BUCKET/producer/log.json
# and records exactly 1,000 emissions, 700 unique event_ids, 300 duplicates.


def checkpoint_1(session):
    try:
        s3c = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            obj = s3c.get_object(Bucket=BUCKET_NAME, Key=PRODUCER_LOG_KEY)
        except ClientError as e:
            code_ = e.response["Error"]["Code"]
            if code_ in ("NoSuchKey", "NoSuchBucket", "404"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Producer log not found at s3://{BUCKET_NAME}/{PRODUCER_LOG_KEY}. "
                        f"Run run_producer() at the end of the Task 1 cell."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        try:
            body = json.loads(obj["Body"].read())
        except Exception as e:
            return CheckpointResult(
                status="fail",
                error_reason=f"Producer log at {PRODUCER_LOG_KEY} is not valid JSON: {e!r}",
            )

        emitted = int(body.get("emitted", 0))
        duplicates = int(body.get("duplicates", 0))
        ids = body.get("ids") or []
        unique_count = len(set(ids))

        if emitted != TOTAL_EVENT_COUNT:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Producer log reports emitted={emitted}, expected {TOTAL_EVENT_COUNT}. "
                    f"Re-run the Task 1 cell; the producer emits {TOTAL_EVENT_COUNT} records."
                ),
            )
        if unique_count != UNIQUE_EVENT_COUNT:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Producer log reports {unique_count} unique event_id values, expected {UNIQUE_EVENT_COUNT}. "
                    f"The provided producer should not be edited; chaos ratio is fixed for the lab."
                ),
            )
        if duplicates != DUPLICATE_EVENT_COUNT:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Producer log reports duplicates={duplicates}, expected {DUPLICATE_EVENT_COUNT}. "
                    f"The provided producer should not be edited; chaos ratio is fixed for the lab."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=repr(e))


check(1, checkpoint_1)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint reads `s3://BUCKET/producer/log.json` and walks three assertions: `emitted == 1000`, `len(set(ids)) == 700`, `duplicates == 300`. Read the failure reason. The most common miss on the first run is forgetting to call `run_producer()` at the end of the Task 1 cell; the bucket and stream exist but no log object was ever written. The second most common miss is creating the stream with the wrong shard count; the producer chunks at 500 records per `put_records` call and a non-1 shard count does not break that, but the cleanup tag check will misfire.

</details>

<details><summary>Hint 2 (direction)</summary>

Four API calls plus one helper call in this task. `s3.create_bucket(Bucket=BUCKET_NAME)` (no `LocationConstraint` in us-east-1) followed by `s3.put_bucket_tagging(Bucket=BUCKET_NAME, Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]})`. `kinesis.create_stream(StreamName=STREAM_NAME, ShardCount=SHARD_COUNT, StreamModeDetails={"StreamMode": "PROVISIONED"}, Tags={LAB_TAG_KEY: LAB_TAG_VALUE})` followed by `kinesis.get_waiter("stream_exists").wait(StreamName=STREAM_NAME)` (waits for ACTIVE). `glue.create_database(DatabaseInput={"Name": GLUE_DB, "LocationUri": f"s3://{BUCKET_NAME}/"})`. `run_athena(CREATE_ICEBERG_SQL, database=GLUE_DB)` to create the Iceberg sink table. Then `run_producer()` to fire the 1,000-record emission and write the log object.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 1:

```python
s3.create_bucket(Bucket=BUCKET_NAME)
s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)

kinesis.create_stream(
    StreamName=STREAM_NAME,
    ShardCount=SHARD_COUNT,
    StreamModeDetails={"StreamMode": "PROVISIONED"},
    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
)
kinesis.get_waiter("stream_exists").wait(StreamName=STREAM_NAME)

glue.create_database(
    DatabaseInput={
        "Name": GLUE_DB,
        "LocationUri": f"s3://{BUCKET_NAME}/",
    }
)

run_athena(CREATE_ICEBERG_SQL, database=GLUE_DB)

run_producer()
```

If `create_stream` returns `AccessDenied`, your `labrun-test` IAM user is missing `AmazonKinesisFullAccess` from the lab page. If `create_database` returns `AccessDenied`, you are missing `AWSGlueServiceRole`. If `run_athena` returns `AccessDenied` on the Athena workgroup, you are missing `AmazonAthenaFullAccess`.

</details>


**Wallet check.** **One meter started.** The Kinesis 1-shard stream is now billing at $0.015 per shard-hour. The S3 bucket, the Glue database, and the Iceberg table are all free at this volume. The producer emission was 1,000 PUT payload units across two `put_records` calls; per-PUT pricing on Kinesis is $0.014 per million ($0.000014 each), so the producer cost a small fraction of a penny. The Athena CREATE TABLE scanned zero data; CREATE statements are not billed. Total session cost so far: under one cent, dominated by the wall-clock that the stream meter has been running.


## Task 2: Stand up DynamoDB, deploy the consumer Lambda, wire it to the stream, and wait for the drain

Now build the consumer that proves exactly-once. Five steps:

1. **Create the DynamoDB on-demand dedupe table.** Hash key `event_id` (string). PAY_PER_REQUEST billing mode. Tagged. The schema is one column plus the hash key; that is all the consumer needs.
2. **Create the Lambda consumer IAM role.** Trust policy: `lambda.amazonaws.com`. Attach the seven managed policies the consumer needs: `AmazonDynamoDBFullAccess`, `AmazonKinesisReadOnlyAccess`, `AWSLambdaBasicExecutionRole`, `AmazonS3FullAccess`, `AmazonAthenaFullAccess`, `AWSGlueServiceRole`, plus `service-role/AWSLambdaKinesisExecutionRole` (the one with the right Kinesis trigger permissions).
3. **Deploy the consumer Lambda.** Python 3.13 runtime. The code zips itself in-memory; you write the handler logic. The handler reads each Kinesis record, attempts `dynamodb.put_item(ConditionExpression="attribute_not_exists(event_id)")`, on `ConditionalCheckFailedException` it logs a specific line and skips, on success it runs an Athena `INSERT INTO events_iceberg VALUES (...)`.
4. **Create the event-source mapping.** `BatchSize=1`, `StartingPosition=TRIM_HORIZON`, `Enabled=True`. BatchSize=1 keeps the conditional-write semantics serial-per-record, which is what Checkpoint 4 measures.
5. **Wait for the consumer to drain the stream.** Two `time.sleep` blocks: 15 seconds for the mapping to reach `Enabled` and start polling, then ~60 seconds for the 1,000 records to land in DynamoDB and Iceberg. Quirky wait messages welcome.

The Lambda handler logic is the meat of the lab. Hint 3 contains the full handler. The four lines that matter:

```python
try:
    ddb.put_item(
        TableName=DEDUPE_TABLE,
        Item={"event_id": {"S": event_id}, "first_seen": {"N": str(int(time.time()))}},
        ConditionExpression="attribute_not_exists(event_id)",
    )
except botocore.exceptions.ClientError as e:
    if e.response["Error"]["Code"] == "ConditionalCheckFailedException":
        print(f"DEDUPE_REJECT event_id={event_id} reason=ConditionalCheckFailedException")
        return  # skip the Iceberg write
    raise
# DynamoDB write succeeded; safe to write to Iceberg
athena_insert(event_id, customer_id, amount_cents, event_timestamp, batch_id)
```

The print line that contains `ConditionalCheckFailedException` is what Checkpoint 4 queries CloudWatch Logs Insights for. Do not change the substring.

Checkpoint 2 scans the DynamoDB table and asserts `Count == 700`.


In [ ]:
# NBVAL_SKIP
# Task 2: DynamoDB dedupe table, consumer Lambda IAM role, consumer Lambda,
# event-source mapping, then wait for the drain.

import textwrap

iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
lambda_client = boto3.client(
    "lambda",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
dynamodb = boto3.client(
    "dynamodb",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

stream_arn = f"arn:aws:kinesis:{REGION}:{ACCOUNT_ID}:stream/{STREAM_NAME}"

consumer_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "lambda.amazonaws.com"},
        "Action": "sts:AssumeRole",
    }],
}

CONSUMER_MANAGED_POLICIES = [
    "arn:aws:iam::aws:policy/AmazonDynamoDBFullAccess",
    "arn:aws:iam::aws:policy/AmazonKinesisReadOnlyAccess",
    "arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
    "arn:aws:iam::aws:policy/AmazonS3FullAccess",
    "arn:aws:iam::aws:policy/AmazonAthenaFullAccess",
    "arn:aws:iam::aws:policy/service-role/AWSGlueServiceRole",
    "arn:aws:iam::aws:policy/service-role/AWSLambdaKinesisExecutionRole",
]

# YOUR CODE: create the DynamoDB table with KeySchema=[{"AttributeName":
# "event_id", "KeyType": "HASH"}], AttributeDefinitions=[{"AttributeName":
# "event_id", "AttributeType": "S"}], BillingMode="PAY_PER_REQUEST",
# Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]. Then wait for ACTIVE
# via dynamodb.get_waiter("table_exists").wait(TableName=DEDUPE_TABLE).

# YOUR CODE: create the consumer IAM role with the trust policy above,
# tag it, then loop CONSUMER_MANAGED_POLICIES and call
# iam.attach_role_policy(RoleName=CONSUMER_ROLE_NAME, PolicyArn=arn) for each.

consumer_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{CONSUMER_ROLE_NAME}"


# Consumer Lambda source: zip in memory. The handler is the heart of the lab.
# It reads each Kinesis record, attempts the conditional write to DynamoDB,
# logs DEDUPE_REJECT on ConditionalCheckFailedException, otherwise fires the
# Iceberg INSERT through Athena.
CONSUMER_SOURCE = textwrap.dedent(f"""
    import base64
    import json
    import os
    import time

    import boto3
    import botocore

    REGION = os.environ["REGION"]
    DEDUPE_TABLE = os.environ["DEDUPE_TABLE"]
    GLUE_DB = os.environ["GLUE_DB"]
    ICEBERG_TABLE = os.environ["ICEBERG_TABLE"]
    BUCKET = os.environ["BUCKET"]

    ddb = boto3.client("dynamodb", region_name=REGION)
    athena = boto3.client("athena", region_name=REGION)


    def athena_insert(event_id, customer_id, amount_cents, event_timestamp, batch_id):
        sql = (
            f"INSERT INTO {{ICEBERG_TABLE}} VALUES ("
            f"'{{event_id}}', {{customer_id}}, {{amount_cents}}, "
            f"'{{event_timestamp}}', '{{batch_id}}')"
        )
        resp = athena.start_query_execution(
            QueryString=sql,
            QueryExecutionContext={{"Database": GLUE_DB}},
            ResultConfiguration={{"OutputLocation": f"s3://{{BUCKET}}/athena-results/"}},
        )
        qid = resp["QueryExecutionId"]
        deadline = time.time() + 60
        while time.time() < deadline:
            desc = athena.get_query_execution(QueryExecutionId=qid)
            state = desc["QueryExecution"]["Status"]["State"]
            if state == "SUCCEEDED":
                return
            if state in ("FAILED", "CANCELLED"):
                reason = desc["QueryExecution"]["Status"].get("StateChangeReason", "")
                raise RuntimeError(f"Athena INSERT {{state}}: {{reason}}")
            time.sleep(1)
        raise RuntimeError("Athena INSERT timed out after 60s")


    def handler(event, context):
        for record in event.get("Records", []):
            data = json.loads(base64.b64decode(record["kinesis"]["data"]).decode())
            event_id = data["event_id"]
            customer_id = int(data["customer_id"])
            amount_cents = int(data["amount_cents"])
            event_timestamp = data["event_timestamp"]
            batch_id = data["batch_id"]

            try:
                ddb.put_item(
                    TableName=DEDUPE_TABLE,
                    Item={{
                        "event_id": {{"S": event_id}},
                        "first_seen": {{"N": str(int(time.time()))}},
                    }},
                    ConditionExpression="attribute_not_exists(event_id)",
                )
            except botocore.exceptions.ClientError as e:
                if e.response["Error"]["Code"] == "ConditionalCheckFailedException":
                    print(f"DEDUPE_REJECT event_id={{event_id}} reason=ConditionalCheckFailedException")
                    continue
                raise

            athena_insert(event_id, customer_id, amount_cents, event_timestamp, batch_id)

        return {{"processed": len(event.get("Records", []))}}
""").strip()


# Build the in-memory zip with index.py.
zip_buf = io.BytesIO()
with zipfile.ZipFile(zip_buf, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("index.py", CONSUMER_SOURCE)
zip_bytes = zip_buf.getvalue()

# Retry create_function until IAM role propagation lands (10-15 seconds).
def _create_consumer_with_retry() -> dict:
    last_err = None
    for attempt in range(12):
        try:
            return lambda_client.create_function(
                FunctionName=CONSUMER_LAMBDA_NAME,
                Runtime=LAMBDA_RUNTIME,
                Role=consumer_role_arn,
                Handler=LAMBDA_HANDLER,
                Code={"ZipFile": zip_bytes},
                Timeout=60,
                MemorySize=256,
                Environment={"Variables": {
                    "REGION": REGION,
                    "DEDUPE_TABLE": DEDUPE_TABLE,
                    "GLUE_DB": GLUE_DB,
                    "ICEBERG_TABLE": ICEBERG_TABLE,
                    "BUCKET": BUCKET_NAME,
                }},
                Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
            )
        except ClientError as e:
            code_ = e.response["Error"]["Code"]
            if code_ in ("InvalidParameterValueException", "AccessDeniedException"):
                last_err = e
                msg = ["role still propagating, waiting", "patience pays the bill, hold", "the principal is still being convinced"][attempt % 3]
                print(f"  attempt {attempt + 1}: {msg}... ({code_})")
                time.sleep(5)
                continue
            raise
    raise RuntimeError(f"Lambda create_function never succeeded; last error: {last_err}")


# YOUR CODE: call _create_consumer_with_retry() to deploy the Lambda. The
# helper handles the IAM propagation retry loop for you.

# YOUR CODE: create the event-source mapping with EventSourceArn=stream_arn,
# FunctionName=CONSUMER_LAMBDA_NAME, BatchSize=1, StartingPosition=
# "TRIM_HORIZON", Enabled=True. Capture the UUID into EVENT_MAPPING_UUID
# (module-level) and update CLEANUP_MANIFEST so the cleanup cell can delete
# the mapping. Hint 3 has the two-line update.

# Wait for the mapping to reach Enabled and then for the consumer to drain
# the 1,000 records into DynamoDB and Iceberg.
print("Waiting for the event-source mapping to start polling...")
time.sleep(15)
print("Waiting for the consumer to drain the stream. 1,000 records, BatchSize=1,")
print("each invocation fires an Athena INSERT against the Iceberg sink.")
print("Coffee break time. The drain takes roughly 90 to 180 seconds depending")
print("on Athena's mood.")
drain_deadline = time.time() + 240
while time.time() < drain_deadline:
    try:
        scan = dynamodb.scan(TableName=DEDUPE_TABLE, Select="COUNT")
        count = scan.get("Count", 0)
    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceNotFoundException":
            print("  DynamoDB table not visible yet (table_exists waiter may not have flushed)")
            time.sleep(5)
            continue
        raise
    print(f"  DynamoDB dedupe rows so far: {count} / {UNIQUE_EVENT_COUNT}")
    if count >= UNIQUE_EVENT_COUNT:
        print("Drain complete.")
        break
    time.sleep(10)
else:
    print(f"  drain did not finish in 240s; DynamoDB rows: {count}. Checkpoint 2 will report exact state.")


In [ ]:
# NBVAL_SKIP
# Checkpoint 2: DynamoDB dedupe table contains exactly 700 rows
# (one per unique event_id; the 300 duplicate records were rejected
# by the ConditionExpression="attribute_not_exists(event_id)" guard).


def checkpoint_2(session):
    try:
        ddb = boto3.client(
            "dynamodb",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            resp = ddb.scan(TableName=DEDUPE_TABLE, Select="COUNT")
        except ClientError as e:
            if e.response["Error"]["Code"] == "ResourceNotFoundException":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"DynamoDB table {DEDUPE_TABLE!r} does not exist. "
                        f"Run the Task 2 cell to create it."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        count = int(resp.get("Count", 0))
        if count == 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"DynamoDB dedupe table is empty. The consumer Lambda has not "
                    f"written anything. Check that the event-source mapping is "
                    f"Enabled (lambda.get_event_source_mapping(UUID=...)) and the "
                    f"consumer log group exists at /aws/lambda/{CONSUMER_LAMBDA_NAME}."
                ),
            )
        if count != UNIQUE_EVENT_COUNT:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"DynamoDB dedupe row count is {count}, expected {UNIQUE_EVENT_COUNT}. "
                    f"If {count} > {UNIQUE_EVENT_COUNT}, the consumer is not running the "
                    f"conditional write correctly (ConditionExpression= "
                    f"'attribute_not_exists(event_id)'). If {count} < {UNIQUE_EVENT_COUNT}, "
                    f"the consumer has not finished draining; re-run this checkpoint in "
                    f"30-60 seconds."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=repr(e))


check(2, checkpoint_2)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint scans the DynamoDB dedupe table with `Select="COUNT"` and asserts the count equals 700. Read the failure reason. If the count is 0, the consumer Lambda is not running; check the event-source mapping state via `lambda_client.get_event_source_mapping(UUID=EVENT_MAPPING_UUID)` and confirm `State == "Enabled"`. If the count is greater than 700, the conditional write is not engaging; check the handler's `ConditionExpression` is exactly `"attribute_not_exists(event_id)"`. If the count is less than 700, the drain is not finished; wait another 30 to 60 seconds and re-run the checkpoint.

</details>

<details><summary>Hint 2 (direction)</summary>

Five API call clusters in this task. `dynamodb.create_table(TableName=DEDUPE_TABLE, KeySchema=[{"AttributeName": "event_id", "KeyType": "HASH"}], AttributeDefinitions=[{"AttributeName": "event_id", "AttributeType": "S"}], BillingMode="PAY_PER_REQUEST", Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}])` then `dynamodb.get_waiter("table_exists").wait(TableName=DEDUPE_TABLE)`. `iam.create_role(RoleName=CONSUMER_ROLE_NAME, AssumeRolePolicyDocument=json.dumps(consumer_trust_policy), Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}])` then loop the managed policies with `iam.attach_role_policy(RoleName=CONSUMER_ROLE_NAME, PolicyArn=arn)`. `_create_consumer_with_retry()` to deploy the Lambda. `esm = lambda_client.create_event_source_mapping(EventSourceArn=stream_arn, FunctionName=CONSUMER_LAMBDA_NAME, BatchSize=1, StartingPosition="TRIM_HORIZON", Enabled=True)` then `EVENT_MAPPING_UUID = esm["UUID"]` and update `CLEANUP_MANIFEST[1] = CleanupResource(type="lambda_event_source_mapping", id=EVENT_MAPPING_UUID, region=REGION, cli_delete_command=f"aws lambda delete-event-source-mapping --uuid {EVENT_MAPPING_UUID}")`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 2:

```python
dynamodb.create_table(
    TableName=DEDUPE_TABLE,
    KeySchema=[{"AttributeName": "event_id", "KeyType": "HASH"}],
    AttributeDefinitions=[{"AttributeName": "event_id", "AttributeType": "S"}],
    BillingMode="PAY_PER_REQUEST",
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
dynamodb.get_waiter("table_exists").wait(TableName=DEDUPE_TABLE)

iam.create_role(
    RoleName=CONSUMER_ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(consumer_trust_policy),
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
for policy_arn in CONSUMER_MANAGED_POLICIES:
    iam.attach_role_policy(RoleName=CONSUMER_ROLE_NAME, PolicyArn=policy_arn)

_create_consumer_with_retry()

esm = lambda_client.create_event_source_mapping(
    EventSourceArn=stream_arn,
    FunctionName=CONSUMER_LAMBDA_NAME,
    BatchSize=1,
    StartingPosition="TRIM_HORIZON",
    Enabled=True,
)
EVENT_MAPPING_UUID = esm["UUID"]
CLEANUP_MANIFEST[1] = CleanupResource(
    type="lambda_event_source_mapping",
    id=EVENT_MAPPING_UUID,
    region=REGION,
    cli_delete_command=f"aws lambda delete-event-source-mapping --uuid {EVENT_MAPPING_UUID}",
)
```

The Lambda handler is the actual mechanism that makes exactly-once work. The provided `CONSUMER_SOURCE` string contains the full handler. If you want to inspect it before deployment, `print(CONSUMER_SOURCE)` in a scratch cell.

The two non-obvious bits in the handler:

- `ConditionExpression="attribute_not_exists(event_id)"` is the idempotency primitive. DynamoDB rejects the `put_item` if the row already exists; the consumer logs `DEDUPE_REJECT` and `continue`s to the next record.
- The Iceberg `INSERT` fires only after the DynamoDB write succeeds. Ordering matters; if the Iceberg write fired first and then the DynamoDB write failed on the next batch retry, you would have an orphan Iceberg row.

If `create_function` returns `InvalidParameterValueException` repeatedly, IAM role propagation is taking longer than 60 seconds (rare; usually 10-15 seconds). Wait another minute and re-run the cell.

</details>


**Wallet check.** Kinesis meter still running at $0.015 per shard-hour. DynamoDB on-demand: ~1,000 writes at the standard rate of $1.25 per million write request units is roughly one tenth of a cent. Lambda invocations: 1,000 at 128-256 MB for under a second each, well inside the always-free 400k GB-seconds monthly tier. The Athena INSERTs against the Iceberg sink are the largest line item: each INSERT scans a tiny amount of data but Athena rounds up to a minimum, so figure 1,000 small INSERTs at fractions of a cent each. Total session cost so far: probably 50 to 90 cents, dominated by the Athena INSERTs.


## Task 3: Confirm the Iceberg sink contains exactly 700 rows with zero duplicate event_ids

The consumer drained the stream in Task 2. The DynamoDB conditional write rejected 300 records (the duplicates); the Iceberg `INSERT` only fired for the 700 unique records. Now you prove it.

This task has no new resource creation. You fire two Athena queries against the Iceberg sink:

```sql
SELECT count(*) FROM events_iceberg
SELECT count(DISTINCT event_id) FROM events_iceberg
```

Both should return 700. If the first returns 1000, the consumer is writing to Iceberg before the DynamoDB write succeeds (ordering bug). If the second is less than the first, the consumer is double-writing for some reason (which the conditional write should prevent; this would mean the handler is catching `ConditionalCheckFailedException` and continuing to the Iceberg write rather than `continue`ing to the next record).

If you ran Task 2 cleanly, both numbers are 700 and Checkpoint 3 passes on first try.


In [ ]:
# NBVAL_SKIP
# Task 3: trigger a 10-second settle, then read the two row counts so the
# values are warm when the checkpoint queries them.

athena = boto3.client(
    "athena",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)


def _run_athena_count(sql: str) -> int:
    resp = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={"Database": GLUE_DB},
        ResultConfiguration={"OutputLocation": f"s3://{BUCKET_NAME}/athena-results/"},
    )
    qid = resp["QueryExecutionId"]
    deadline = time.time() + 60
    while time.time() < deadline:
        desc = athena.get_query_execution(QueryExecutionId=qid)
        state = desc["QueryExecution"]["Status"]["State"]
        if state == "SUCCEEDED":
            results = athena.get_query_results(QueryExecutionId=qid)
            rows = results["ResultSet"]["Rows"]
            # Row 0 is the header; row 1 is the count.
            return int(rows[1]["Data"][0]["VarCharValue"])
        if state in ("FAILED", "CANCELLED"):
            reason = desc["QueryExecution"]["Status"].get("StateChangeReason", "")
            raise RuntimeError(f"Athena query {state}: {reason}\nQuery: {sql}")
        time.sleep(1)
    raise RuntimeError(f"Athena query timed out after 60s: {sql}")


print("Letting the Iceberg sink settle for 10 seconds (catalog visibility)...")
time.sleep(10)

ICEBERG_TOTAL = _run_athena_count(f"SELECT count(*) FROM {ICEBERG_TABLE}")
ICEBERG_DISTINCT = _run_athena_count(f"SELECT count(DISTINCT event_id) FROM {ICEBERG_TABLE}")

print(f"Iceberg total row count: {ICEBERG_TOTAL}")
print(f"Iceberg distinct event_id count: {ICEBERG_DISTINCT}")
print(f"Expected for both: {UNIQUE_EVENT_COUNT}")


In [ ]:
# NBVAL_SKIP
# Checkpoint 3: Iceberg sink has exactly 700 rows AND 700 distinct event_ids.


def checkpoint_3(session):
    try:
        athena_c = boto3.client(
            "athena",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        def _count(sql: str) -> int:
            resp = athena_c.start_query_execution(
                QueryString=sql,
                QueryExecutionContext={"Database": GLUE_DB},
                ResultConfiguration={"OutputLocation": f"s3://{BUCKET_NAME}/athena-results/"},
            )
            qid = resp["QueryExecutionId"]
            deadline = time.time() + 60
            while time.time() < deadline:
                desc = athena_c.get_query_execution(QueryExecutionId=qid)
                state = desc["QueryExecution"]["Status"]["State"]
                if state == "SUCCEEDED":
                    results = athena_c.get_query_results(QueryExecutionId=qid)
                    rows = results["ResultSet"]["Rows"]
                    return int(rows[1]["Data"][0]["VarCharValue"])
                if state in ("FAILED", "CANCELLED"):
                    reason = desc["QueryExecution"]["Status"].get("StateChangeReason", "")
                    raise RuntimeError(f"Athena {state}: {reason}")
                time.sleep(1)
            raise RuntimeError(f"Athena timed out: {sql}")

        try:
            total = _count(f"SELECT count(*) FROM {ICEBERG_TABLE}")
            distinct = _count(f"SELECT count(DISTINCT event_id) FROM {ICEBERG_TABLE}")
        except Exception as e:
            return CheckpointResult(status="error", error_reason=repr(e))

        if total != UNIQUE_EVENT_COUNT:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Iceberg sink has {total} total rows, expected {UNIQUE_EVENT_COUNT}. "
                    f"If {total} > {UNIQUE_EVENT_COUNT}, the handler is writing to "
                    f"Iceberg even when the DynamoDB conditional write rejected the "
                    f"record (the handler must `continue` on ConditionalCheckFailedException, "
                    f"not fall through). If {total} < {UNIQUE_EVENT_COUNT}, the consumer "
                    f"has not finished; wait 30 to 60 seconds and re-run."
                ),
            )
        if distinct != UNIQUE_EVENT_COUNT:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Iceberg sink has {distinct} distinct event_id values, expected "
                    f"{UNIQUE_EVENT_COUNT}. Distinct < total means duplicate event_ids "
                    f"are present, which the conditional write was supposed to prevent. "
                    f"Inspect the handler's exception handling around put_item."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=repr(e))


check(3, checkpoint_3)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint fires two Athena queries against the Iceberg sink: `SELECT count(*) FROM events_iceberg` and `SELECT count(DISTINCT event_id) FROM events_iceberg`. Both must return 700. Read the failure reason. If `count(*) > 700` the handler is writing to Iceberg on duplicate records; the `ConditionalCheckFailedException` branch must `continue` (or `return` if BatchSize is 1, which it is) and NOT fall through to `athena_insert`. If `count(*) < 700`, the consumer is still draining; wait another minute.

</details>

<details><summary>Hint 2 (direction)</summary>

No new API calls in this task. The work is in the Lambda handler from Task 2. The ordering in the handler is the load-bearing detail: DynamoDB `put_item` with `ConditionExpression` runs first; only on success does `athena_insert` fire. On `ConditionalCheckFailedException`, the handler logs `DEDUPE_REJECT` and `continue`s past the Athena write. If you reversed the order (Athena first, DynamoDB second), the Iceberg sink would have all 1,000 records and `count(DISTINCT event_id)` would still be 700; the validator catches the first ordering, not the second.

</details>

<details><summary>Hint 3 (spoiler)</summary>

There is no student-written code for Task 3. The Task 2 handler is what makes Checkpoint 3 pass. For reference, here is the exception-handling branch the handler must contain:

```python
try:
    ddb.put_item(
        TableName=DEDUPE_TABLE,
        Item={"event_id": {"S": event_id}, "first_seen": {"N": str(int(time.time()))}},
        ConditionExpression="attribute_not_exists(event_id)",
    )
except botocore.exceptions.ClientError as e:
    if e.response["Error"]["Code"] == "ConditionalCheckFailedException":
        print(f"DEDUPE_REJECT event_id={event_id} reason=ConditionalCheckFailedException")
        continue  # skip the Iceberg write
    raise
# DynamoDB write succeeded; safe to write to Iceberg
athena_insert(event_id, customer_id, amount_cents, event_timestamp, batch_id)
```

If Checkpoint 3 fails with `count(*) > 700`, your handler is missing the `continue` (or the equivalent `return` when BatchSize is 1) on the rejection branch.

</details>


**Wallet check.** Kinesis meter still running at $0.015 per shard-hour. The two Athena `count(*)` queries scan the Iceberg metadata only, not the row data: cost is fractions of a cent each. Total session cost so far: 60 cents to $1.10, still dominated by the Task 2 Iceberg INSERTs.


## Task 4: Prove the 300 duplicates were rejected by querying CloudWatch Logs Insights

The DynamoDB count says 700. The Iceberg count says 700. But you also want the auditable proof: the consumer logged a `DEDUPE_REJECT` line every time the conditional write rejected a duplicate. CloudWatch Logs Insights lets you query the consumer log group for that specific substring and confirm the count.

This task fires one CloudWatch Logs Insights query against `/aws/lambda/labrun-kinesis-exactly-once-consumer` for the window covering the consumer's runtime. The query searches for `ConditionalCheckFailedException` (the substring inside the handler's print line). The result count should be in the 290 to 310 range; some slack on the upper bound because Lambda may retry a batch if the function crashes mid-batch (re-running the conditional write, which fails again, and re-logs).

Checkpoint 4 asserts `290 <= count <= 310`.

The Logs Insights query string:

```
fields @timestamp, @message
| filter @message like /ConditionalCheckFailedException/
| stats count() as rejections
```

The fire-and-poll pattern is `start_query` followed by `get_query_results` polling until `status == "Complete"`. CloudWatch Logs Insights typically returns in 5-15 seconds for a result this small.


In [ ]:
# NBVAL_SKIP
# Task 4: query CloudWatch Logs Insights for the count of DEDUPE_REJECT lines.

logs = boto3.client(
    "logs",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

CONSUMER_LOG_GROUP = f"/aws/lambda/{CONSUMER_LAMBDA_NAME}"
REJECTION_QUERY = (
    "fields @timestamp, @message\n"
    "| filter @message like /ConditionalCheckFailedException/\n"
    "| stats count() as rejections"
)


def count_rejections() -> int:
    """Fire the Logs Insights query and poll until Complete; return the count.

    Querying back 30 minutes is a safe window for a lab session.
    """
    now = int(time.time())
    start_time = now - 1800
    # YOUR CODE: call logs.start_query(logGroupName=CONSUMER_LOG_GROUP,
    # startTime=start_time, endTime=now, queryString=REJECTION_QUERY). Capture
    # the queryId out of the response. Hint 3 has the call.
    raise NotImplementedError("Replace this with logs.start_query and a poll loop")


# YOUR CODE: assign rejections = count_rejections() and print the value.
# The expected value is in the 290-310 range. Checkpoint 4 validates the
# count is within that window.


In [ ]:
# NBVAL_SKIP
# Checkpoint 4: CloudWatch Logs Insights reports between 290 and 310
# ConditionalCheckFailedException rejections from the consumer Lambda.


def checkpoint_4(session):
    try:
        logs_c = boto3.client(
            "logs",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            logs_c.describe_log_groups(logGroupNamePrefix=CONSUMER_LOG_GROUP)
        except ClientError as e:
            return CheckpointResult(status="error", error_reason=str(e))

        now = int(time.time())
        start_time = now - 1800
        try:
            qresp = logs_c.start_query(
                logGroupName=CONSUMER_LOG_GROUP,
                startTime=start_time,
                endTime=now,
                queryString=REJECTION_QUERY,
            )
        except ClientError as e:
            if e.response["Error"]["Code"] in ("ResourceNotFoundException",):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"CloudWatch log group {CONSUMER_LOG_GROUP!r} does not exist. "
                        f"The consumer Lambda has not been invoked yet; check the "
                        f"event-source mapping state in Task 2."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        qid = qresp["queryId"]
        deadline = time.time() + 60
        rejections = None
        while time.time() < deadline:
            r = logs_c.get_query_results(queryId=qid)
            status_ = r.get("status")
            if status_ == "Complete":
                rows = r.get("results", [])
                if not rows:
                    rejections = 0
                else:
                    for col in rows[0]:
                        if col.get("field") == "rejections":
                            rejections = int(col.get("value", "0"))
                            break
                    if rejections is None:
                        rejections = 0
                break
            if status_ in ("Failed", "Cancelled"):
                return CheckpointResult(
                    status="error",
                    error_reason=f"CloudWatch Logs Insights query {status_}.",
                )
            time.sleep(2)

        if rejections is None:
            return CheckpointResult(
                status="error",
                error_reason="CloudWatch Logs Insights query did not complete in 60s.",
            )

        if rejections < 290:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Consumer log shows {rejections} ConditionalCheckFailedException "
                    f"rejections; expected 290-310. The handler may not be logging on "
                    f"the rejection branch. Confirm the print line in the handler is "
                    f"exactly: print(f\"DEDUPE_REJECT event_id={{event_id}} reason="
                    f"ConditionalCheckFailedException\")."
                ),
            )
        if rejections > 310:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Consumer log shows {rejections} ConditionalCheckFailedException "
                    f"rejections; expected 290-310. More than 310 suggests the Lambda "
                    f"is retrying batches that crashed mid-processing, re-running the "
                    f"conditional write and re-logging. Inspect the consumer log group "
                    f"for exceptions raised in the handler after the DynamoDB write."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=repr(e))


check(4, checkpoint_4)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint runs a CloudWatch Logs Insights query against the consumer log group for the substring `ConditionalCheckFailedException` and asserts the count is in `[290, 310]`. Read the failure reason. If the count is 0, the consumer is not logging the rejection (the print line in the handler is missing or wrong). If the count is much higher than 310, Lambda is retrying batches that crashed mid-record, re-firing the conditional write and re-logging; check the consumer log group for raised exceptions.

</details>

<details><summary>Hint 2 (direction)</summary>

Three calls in this task. `qresp = logs.start_query(logGroupName=CONSUMER_LOG_GROUP, startTime=start_time, endTime=now, queryString=REJECTION_QUERY)` returns a `queryId`. Then a poll loop: `r = logs.get_query_results(queryId=qresp["queryId"])` and check `r["status"] == "Complete"`. When complete, the result shape is `r["results"][0][i]["field"]` and `r["results"][0][i]["value"]` for each named field; the field named `"rejections"` holds the count as a string. Convert with `int()`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 4:

```python
def count_rejections() -> int:
    now = int(time.time())
    start_time = now - 1800
    qresp = logs.start_query(
        logGroupName=CONSUMER_LOG_GROUP,
        startTime=start_time,
        endTime=now,
        queryString=REJECTION_QUERY,
    )
    qid = qresp["queryId"]
    while True:
        r = logs.get_query_results(queryId=qid)
        if r["status"] == "Complete":
            for row in r.get("results", []):
                for col in row:
                    if col.get("field") == "rejections":
                        return int(col.get("value", "0"))
            return 0
        if r["status"] in ("Failed", "Cancelled"):
            raise RuntimeError(f"CloudWatch Logs Insights {r['status']}")
        time.sleep(2)


rejections = count_rejections()
print(f"ConditionalCheckFailedException count from CloudWatch Logs Insights: {rejections}")
print(f"Expected window: {DUPLICATE_EVENT_COUNT - 10} to {DUPLICATE_EVENT_COUNT + 10}")
```

If `start_query` returns `MalformedQueryException`, the query string has a syntax error; the multi-line string above uses `|` as the line separator. If `get_query_results` reports `"Complete"` but the `results` list is empty, the log group exists but no matching lines were found; check the handler's print line.

</details>


**Wallet check.** Kinesis meter still running at $0.015 per shard-hour. CloudWatch Logs Insights queries are billed at $0.005 per GB of log data scanned; this lab's consumer log group holds maybe 1 MB, so the query cost is sub-penny. Total session cost so far: 70 cents to $1.30. The cleanup cell below stops the Kinesis meter the moment delete returns; everything else is at-rest free or pay-per-request with no idle cost.


In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation,
# critical-first order per RESOURCE-SAFETY-SPEC Rule 2.
#
# labrun-checks 0.8.0 ships AWS adapters for every resource type in this
# manifest, including iceberg_table (DROP TABLE via Athena). The _LabAdapter
# wrapper below predates the 0.8.0 iceberg_table adapter and is now
# functionally equivalent to the default AwsCleanupAdapter; a follow-up
# can drop it and call run_cleanup against the bundled adapter directly.

import sys
from labrun_checks.adapters.aws import AwsCleanupAdapter


class _LabAdapter:
    """AwsCleanupAdapter wrapper that adds Lab 11 iceberg_table teardown.

    iceberg_table cleanup fires `DROP TABLE IF EXISTS <db>.<table>` via Athena.
    Iceberg metadata files live under s3://BUCKET/events_iceberg/; the
    subsequent s3_bucket delete in the manifest handles the data prefix
    cleanup as part of the bucket-empty step.
    """

    def __init__(self) -> None:
        self._aws = AwsCleanupAdapter()

    def _athena_client(self, credentials: dict):
        return boto3.client(
            "athena",
            aws_access_key_id=credentials["aws_access_key_id"],
            aws_secret_access_key=credentials["aws_secret_access_key"],
            aws_session_token=credentials.get("aws_session_token"),
            region_name=credentials.get("region", REGION),
        )

    def delete_resource(self, credentials: dict, resource) -> None:
        if resource.type == "iceberg_table":
            athena_c = self._athena_client(credentials)
            db, _, tbl = resource.id.partition(".")
            try:
                qresp = athena_c.start_query_execution(
                    QueryString=f"DROP TABLE IF EXISTS {tbl}",
                    QueryExecutionContext={"Database": db},
                    ResultConfiguration={"OutputLocation": f"s3://{BUCKET_NAME}/athena-results/"},
                )
                qid = qresp["QueryExecutionId"]
                deadline = time.time() + 60
                while time.time() < deadline:
                    desc = athena_c.get_query_execution(QueryExecutionId=qid)
                    state = desc["QueryExecution"]["Status"]["State"]
                    if state == "SUCCEEDED":
                        return
                    if state in ("FAILED", "CANCELLED"):
                        # Already-gone DROP errors are fine; surface only real failures.
                        reason = desc["QueryExecution"]["Status"].get("StateChangeReason", "")
                        if "does not exist" in reason.lower() or "not found" in reason.lower():
                            return
                        raise RuntimeError(f"DROP TABLE {state}: {reason}")
                    time.sleep(1)
                return
            except ClientError as e:
                if e.response["Error"]["Code"] in (
                    "EntityNotFoundException", "ResourceNotFoundException",
                ):
                    return
                raise

        return self._aws.delete_resource(credentials, resource)

    def describe_resource(self, credentials: dict, resource) -> bool:
        if resource.type == "iceberg_table":
            glue_c = boto3.client(
                "glue",
                aws_access_key_id=credentials["aws_access_key_id"],
                aws_secret_access_key=credentials["aws_secret_access_key"],
                aws_session_token=credentials.get("aws_session_token"),
                region_name=credentials.get("region", REGION),
            )
            db, _, tbl = resource.id.partition(".")
            try:
                glue_c.get_table(DatabaseName=db, Name=tbl)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] in (
                    "EntityNotFoundException", "ResourceNotFoundException",
                ):
                    return False
                raise

        return self._aws.describe_resource(credentials, resource)

    def tag_scan(self, credentials: dict, lab_slug: str, region: str) -> list[str]:
        return self._aws.tag_scan(credentials, lab_slug, region)


# Empty the S3 bucket before run_cleanup tears it down. S3 buckets cannot
# be deleted while they contain objects, and the Iceberg sink + Athena
# results + producer log are all in this bucket.
print("Emptying the S3 bucket before teardown...")
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
try:
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=BUCKET_NAME):
        keys = [{"Key": obj["Key"]} for obj in page.get("Contents", [])]
        if keys:
            s3.delete_objects(Bucket=BUCKET_NAME, Delete={"Objects": keys})
except ClientError as e:
    if e.response["Error"]["Code"] != "NoSuchBucket":
        print(f"Bucket emptying ran into: {e}. Continuing to cleanup.")

# Detach managed policies from the consumer role before role-delete; the
# iam_role adapter handles detach for attached policies, but we surface
# any non-already-gone errors here so they do not silently block the role
# delete.
iam_client = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
try:
    attached = iam_client.list_attached_role_policies(RoleName=CONSUMER_ROLE_NAME)
    for p in attached.get("AttachedPolicies", []):
        try:
            iam_client.detach_role_policy(
                RoleName=CONSUMER_ROLE_NAME, PolicyArn=p["PolicyArn"]
            )
        except ClientError as e:
            if e.response["Error"]["Code"] != "NoSuchEntity":
                print(f"Detach for {p['PolicyArn']} ran into: {e}. Continuing.")
except ClientError as e:
    if e.response["Error"]["Code"] != "NoSuchEntity":
        print(f"List policies for {CONSUMER_ROLE_NAME} ran into: {e}. Continuing.")

result = run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter())

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = sum(1 for r in CLEANUP_MANIFEST if getattr(r, "critical", False))
standard_total = len(CLEANUP_MANIFEST) - critical_total
critical_destroyed = critical_total - sum(
    1 for r in CLEANUP_MANIFEST if getattr(r, "critical", False) and r.id in failed_ids
)
standard_destroyed = standard_total - sum(
    1 for r in CLEANUP_MANIFEST if not getattr(r, "critical", False) and r.id in failed_ids
)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)


**Session total: $1.00 to $2.00.** One hourly meter this lab: the Kinesis 1-shard stream at $0.015 per shard-hour, which runs for the ~75-minute hands-on window plus the 5 to 10 minutes of cleanup wall-clock, so about 2 cents total. The expensive line item is the 1,000 small Athena INSERTs the consumer fires against the Iceberg sink: each INSERT pays the Athena per-query minimum (1 MB minimum data scanned at $5/TB, so about $0.000005 per query plus a per-DML overhead), and 1,000 of them lands somewhere between 80 cents and $1.80 depending on the day's Iceberg metadata behavior. DynamoDB on-demand is pay-per-request: 1,000 writes is one tenth of a cent. Lambda invocations are inside the always-free tier. CloudWatch Logs Insights query is fractions of a cent.

The cert-canonical version of this pattern (Kinesis Data Streams plus a Lambda consumer plus a sink) charges the same shard-hour rate regardless of whether the sink is Redshift, OpenSearch, S3, or Iceberg; the per-record sink cost is what changes. For production exactly-once: a DynamoDB conditional write on a stable id is the cheapest idempotency primitive in AWS at small-to-medium scale; at large scale, teams move the dedupe state into the sink itself (Iceberg merge-on-write, Redshift MERGE) so they only pay one round-trip per record instead of two. The interview talking point holds either way: a stable `event_id` plus an atomic-per-row sink-side dedupe is the only pattern that survives a producer with retries.

Check your AWS Billing console in 24 hours to confirm the cleanup landed the meter at zero.


## These are not graded. They are for you.

Two open prompts and two exam-style MCQs. Sit with them before closing the notebook.

1. **One shard versus ten shards.** The lab runs with `ShardCount=1` and `BatchSize=1` on the event-source mapping, so the conditional writes are effectively serial per `event_id`. Walk through what changes when you run the same pipeline against a 10-shard stream with `BatchSize=100` and `ParallelizationFactor=10`. Per-shard, conditional writes are still atomic in DynamoDB (the `attribute_not_exists` check is a single-row condition). The actual change is in the Iceberg write path: per-record `INSERT` becomes a per-batch `INSERT` of up to 100 records, which the handler now does in one Athena call. Walk through whether the same DynamoDB conditional-write pattern still holds, what (if anything) breaks, and how you would batch the Iceberg writes to drop the Athena round-trip count from 700 to 70.

2. **What if the producer cannot give you a stable event_id?** Some upstream systems emit events with no deterministic id (think raw clickstream beacons or webhook payloads from a third party that does not stamp one for you). Walk through the alternatives: hash the payload (works if the payload is byte-identical across retries, breaks if the producer adds a retry counter), use the Kinesis sequence number as the dedupe key (works only within a single shard's stream segment, breaks on resharding), use a `(producer_id, monotonic_counter)` tuple (works if the producer state survives crashes). For each, identify when it is the right tool and when conditional write on a stable id is still the cleanest answer.

## Exam-Style Practice

**Q1.** Your producer is upgraded to enable EventBridge Pipes auto-retry. Each retry replays the same record with the same `event_id`. The simplest sink-side change to keep exactly-once semantics is:

A) Conditional write on a stable `event_id` (the same pattern this lab built).

B) Sequence-number-based dedup window keyed on the Kinesis sequence number with a 5-minute TTL cache in the consumer.

C) Transactional writes via DynamoDB Transactions (TransactWriteItems) coordinated across the dedupe table and the sink.

D) Retry suppression at the source (disable EventBridge Pipes auto-retry).

<details><summary>Show answer</summary>

**A**.

- A) Right. The producer already emits a stable `event_id`. A conditional `put_item` on `event_id` is one round-trip per record, atomic-per-row in DynamoDB, and survives any number of retries from any upstream system. Adding EventBridge Pipes auto-retry does not change the sink-side contract because the dedupe key is producer-stable, not delivery-stable.
- B) Wrong. Kinesis sequence numbers are per-shard and per-stream-segment; they change on resharding and they do not survive a producer-side retry that goes through EventBridge Pipes (Pipes re-emits with a new sequence number even if `event_id` is the same). A sequence-number cache also requires in-memory state in the consumer, which Lambda warm-pool eviction breaks.
- C) Wrong. `TransactWriteItems` is the right primitive when you need atomicity across multiple writes, not for idempotency. The Iceberg sink is not a DynamoDB item, so a DynamoDB transaction cannot cover both. Adds cost (transactions are 2x the write capacity) and complexity for no idempotency benefit over option A.
- D) Wrong. "Disable the retry" is the worst answer because it shifts the durability problem upstream: the producer's network is flaky, that is the entire premise of the lab. Removing retries makes the pipeline lossy. The cert tests recognition of the principle that durability and exactly-once are sink-side concerns, not source-side concerns.

</details>

**Q2.** You are designing a multi-shard variant of this lab's pipeline. The stream has 20 shards and the consumer's event-source mapping uses `BatchSize=500` and `ParallelizationFactor=10` (so up to 10 concurrent Lambda invocations per shard). You need to keep exactly-once semantics. Which statement is true?

A) The DynamoDB conditional write on `event_id` still holds because the condition is single-row atomic; concurrent invocations attempting the same `event_id` will see exactly one succeed and the others get `ConditionalCheckFailedException`.

B) The DynamoDB conditional write breaks under parallel shards because `attribute_not_exists` is eventually consistent across shards.

C) You need to switch to DynamoDB transactional writes (`TransactWriteItems`) to guarantee atomicity across the dedupe write and the sink write.

D) You need to add a global DynamoDB Streams trigger to fan out the writes serially so the conditional write semantics are preserved.

<details><summary>Show answer</summary>

**A**.

- A) Right. DynamoDB `put_item` with `ConditionExpression` is strongly consistent and atomic at the row level. Whether the call comes from one consumer or twenty parallel consumers, exactly one `put_item` wins for a given `event_id`; the others get `ConditionalCheckFailedException`. The lab's single-shard configuration was for pedagogical simplicity, not because the pattern requires it.
- B) Wrong. DynamoDB's condition checks are strongly consistent, not eventually consistent. The confusion sometimes comes from DynamoDB's read-consistency model (eventually-consistent reads are the default for `get_item`, strongly-consistent reads are opt-in), but condition checks on writes are always strongly consistent.
- C) Wrong. `TransactWriteItems` would let you write to two DynamoDB items atomically (e.g., the dedupe table and a sink table), but the Iceberg sink is not a DynamoDB table, so transactions cannot cover both. You also pay 2x write capacity for transactions; it adds cost without solving exactly-once for a non-DynamoDB sink.
- D) Wrong. Adding a DynamoDB Streams trigger to serialize writes would defeat the purpose of running multiple shards. The conditional-write pattern is concurrency-friendly by design; serializing it throws away throughput for a problem that does not exist.

</details>
